In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).parent.parent))
from datetime import datetime
import pickle
from utils.data_cleaning import run_cleaning_pipeline
from utils.merge_pipeline import CANONICAL_SCHEMA
from utils.csv_export import write_to_csv

In [ ]:
merged_read_location = 'output/merged_df.pkl'
cleaned_write_location = 'output/cleaned_df.pkl'

In [ ]:
with open(merged_read_location, 'rb') as f:
    merged_df = pickle.load(f)
print(f"Loaded merged dataset: {len(merged_df)} rows, {merged_df.shape[1]} columns")
merged_df.head()

### Portuguese Genre Translation Map

Translation map based on recalbox['genre_rom'].unique() — found during analysis in 03-data_analysis.ipynb.

In [ ]:
portuguese_genre_map = {
    # Core genres
    'Ação': 'Action',
    'Plataforma': 'Platform',
    'Quebra-Cabeças': 'Puzzle',
    'Quebra-cabeças': 'Puzzle',
    'Esporte': 'Sports',
    'Aventura': 'Adventure',
    'Luta': 'Fighting',
    'Briga De Rua': "Beat'em Up",
    'Briga de rua': "Beat'em Up",
    'Estratégia': 'Strategy',
    'Simulação': 'Simulation',
    'Corrida, Pilotagem': 'Racing',
    'Jogos De RPG': 'Role Playing Game',
    'Jogos de RPG': 'Role Playing Game',
    'Tiro': 'Shooter',
    'Pinball': 'Pinball',
    'Educacional': 'Educational',
    'Variados': 'Various',
    'Cassino': 'Casino',
    
    # Compound terms
    'Jogo de tabuleiro': 'Board Game',
    'Jogo De Tabuleiro': 'Board Game',
    'Cartas': 'Cards',
    'Corrida Em 3ª Pessoa': 'Racing (Third Person)',
    'Tiro Com Acessórios': 'Light Gun Shooter',
    
    # Platform subtypes
    'Plataforma / Combate Com Rolagem': 'Platform / Fighter Scrolling',
    'Plataforma / Combate com rolagem': 'Platform / Fighter Scrolling',
    'Plataforma / Corre E Pula': 'Platform / Jump & Run',
    'Plataforma / Corre E Pula Com Rolagem': 'Platform / Jump & Run',
    'Combate Com Rolagem': 'Fighter Scrolling',
    'Combate com rolagem': 'Fighter Scrolling',
    'Corre E Pula': 'Jump & Run',
    'Corre E Pula Com Rolagem': 'Jump & Run',
    
    # Action subtypes
    'Ação / Aventura': 'Action / Adventure',
    
    # Sports subtypes
    'Esporte / Futebol': 'Sports / Football (Soccer)',
    'Esporte / Basquete': 'Sports / Basketball',
    'Esporte / Tênis': 'Sports / Tennis',
    'Esporte / Futebol Americano': 'Sports / Football (American)',
    'Esporte / Golfe': 'Sports / Golf',
    'Esporte / Boxe': 'Sports / Boxing',
    'Esporte / Luta Livre': 'Sports / Wrestling',
    'Basquete': 'Basketball',
    'Tênis': 'Tennis',
    'Futebol': 'Football (Soccer)',
    'Futebol Americano': 'Football (American)',
    'Golfe': 'Golf',
    'Boxe': 'Boxing',
    'Luta Livre': 'Wrestling',
}

print(f"Portuguese genre translation map: {len(portuguese_genre_map)} mappings")

### Run Cleaning Pipeline

In [ ]:
cleaned_df = run_cleaning_pipeline(merged_df, genre_translation_map=portuguese_genre_map)
print(f"Cleaning complete: {len(cleaned_df)} rows")

In [ ]:
cleaned_df.head()

In [ ]:
print("Before cleaning:")
print(merged_df.info())
print("\nAfter cleaning:")
print(cleaned_df.info())

In [ ]:
# Genre normalization check
pt_genres_before = merged_df[merged_df['genres'].apply(lambda x: any(w in str(x) for w in ['Ação', 'Plataforma', 'Estratégia', 'Simulação']) if isinstance(x, str) else False)]
pt_genres_after = cleaned_df[cleaned_df['genres'].apply(lambda x: any(w in str(x) for w in ['Ação', 'Plataforma', 'Estratégia', 'Simulação']) if isinstance(x, str) else False)]
print(f"Portuguese genres before: {len(pt_genres_before)}")
print(f"Portuguese genres after: {len(pt_genres_after)}")

In [ ]:
# Release year derivation check
missing_year_before = merged_df['release_year'].isna().sum()
missing_year_after = cleaned_df['release_year'].isna().sum()
print(f"Missing release_year before: {missing_year_before} ({missing_year_before/len(merged_df)*100:.1f}%)")
print(f"Missing release_year after: {missing_year_after} ({missing_year_after/len(cleaned_df)*100:.1f}%)")
print(f"Years derived: {missing_year_before - missing_year_after}")

In [ ]:
# Player parsing check
missing_players_before = merged_df['players'].isna().sum()
missing_players_after = cleaned_df['players'].isna().sum()
print(f"Missing players before: {missing_players_before} ({missing_players_before/len(merged_df)*100:.1f}%)")
print(f"Missing players after: {missing_players_after} ({missing_players_after/len(cleaned_df)*100:.1f}%)")

### Save Cleaned Data

In [ ]:
with open(cleaned_write_location, 'wb') as f:
    pickle.dump(cleaned_df, f)
print(f"Cleaned DataFrame saved to {cleaned_write_location}")

In [ ]:
cleaned_df['version'] = datetime.utcnow().isoformat()
write_to_csv(cleaned_df, Path('output/cleaned_games.csv'), CANONICAL_SCHEMA)
print(f"Cleaned CSV saved to output/cleaned_games.csv ({len(cleaned_df)} rows)")